## Loading the dataset

In [ ]:
import pandas as pd
df=pd.read_csv('/content/judge-1377884607_tweet_product_company.csv', encoding='unicode_escape')

## Data preprocessing

In [ ]:
# Checking for the missing values

df.isna().sum()

,0
tweet_text,1
emotion_in_tweet_is_directed_at,5802
is_there_an_emotion_directed_at_a_brand_or_product,0


In [ ]:
# Removing missing values

df=df.dropna(subset=['tweet_text'])

In [ ]:
# Extract Features and Labels

X=df['tweet_text'].astype(str)
y=df['is_there_an_emotion_directed_at_a_brand_or_product']

In [ ]:
# Maping the labels to the 4 requested classes

label_mapping={
    "No emotion toward brand or product": "neutral",
    "Positive emotion": "positive",
    "Negative emotion": "negative",
    "I can't tell": "no_idea"
}
y_map=y.map(label_mapping)

In [ ]:
# Encoding

import tensorflow as tf
from sklearn.preprocessing import LabelEncoder
encoder=LabelEncoder()
y_encoded=encoder.fit_transform(y_map)
y_categorical=tf.keras.utils.to_categorical(y_encoded,num_classes=4)
print(dict(zip(encoder.classes_,encoder.transform(encoder.classes_))))

{'negative': np.int64(0), 'neutral': np.int64(1), 'no_idea': np.int64(2), 'positive': np.int64(3)}


In [ ]:
# Tokenization and Padding

max_vocab_size=10000
max_sequence_len=50

In [ ]:
# Initialize and fit tokenizer

from tensorflow.keras.preprocessing.text import Tokenizer
tokenizer=Tokenizer(num_words=max_vocab_size,oov_token='<OOV>')
tokenizer.fit_on_texts(X)

In [ ]:
# Converting the text into numerical sequences and padding them to a uniform length

from tensorflow.keras.preprocessing.sequence import pad_sequences
X_seq=tokenizer.texts_to_sequences(X)
X_pad=pad_sequences(X_seq,maxlen=max_sequence_len,padding='post',truncating='post')

## Model Training

In [ ]:
# Train and test split

from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test=train_test_split(X_pad,y_categorical,test_size=0.2,random_state=42)

In [ ]:
# Building the LSTM model

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout
model=Sequential([
    Embedding(input_dim=max_vocab_size,output_dim=64,input_length=max_sequence_len),
    LSTM(64,return_sequences=False),
    Dropout(0.5), # Helps prevent overfitting
    Dense(32,activation='relu'),
    Dense(4,activation='softmax') # 4 Output nodes for our 4 classes
])

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [ ]:
model.compile(loss='categorical_crossentropy',optimizer='adam',metrics=['accuracy'])
model.summary()

Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_2 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_2 (LSTM)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [ ]:
# Training the LSTM model

history=model.fit(X_train,y_train, epochs=10, batch_size=64, validation_split=0.2)

Epoch 1/10
91/91 ━━━━━━━━━━━━━━━━━━━━ 4s 46ms/step - accuracy: 0.5820 - loss: 0.9421 - val_accuracy: 0.6021 - val_loss: 0.9117
Epoch 2/10
91/91 ━━━━━━━━━━━━━━━━━━━━ 3s 36ms/step - accuracy: 0.5906 - loss: 0.9352 - val_accuracy: 0.6021 - val_loss: 0.9185
Epoch 3/10
91/91 ━━━━━━━━━━━━━━━━━━━━ 4s 43ms/step - accuracy: 0.5920 - loss: 0.9331 - val_accuracy: 0.6021 - val_loss: 0.9132
Epoch 4/10
91/91 ━━━━━━━━━━━━━━━━━━━━ 3s 37ms/step - accuracy: 0.5928 - loss: 0.9302 - val_accuracy: 0.6021 - val_loss: 0.9126
Epoch 5/10
91/91 ━━━━━━━━━━━━━━━━━━━━ 3s 36ms/step - accuracy: 0.5925 - loss: 0.9279 - val_accuracy: 0.6021 - val_loss: 0.9117
Epoch 6/10
91/91 ━━━━━━━━━━━━━━━━━━━━ 3s 33ms/step - accuracy: 0.5925 - loss: 0.9270 - val_accuracy: 0.6021 - val_loss: 0.9140
Epoch 7/10
91/91 ━━━━━━━━━━━━━━━━━━━━ 4s 44ms/step - accuracy: 0.5921 - loss: 0.9208 - val_accuracy: 0.5945 - val_loss: 0.8877
Epoch 8/10
91/91 ━━━━━━━━━━━━━━━━━━━━ 3s 36ms/step - accuracy: 0.6260 - loss: 0.8601 - val_accuracy: 0.6289 - v

## Evaluation

In [ ]:
# Evalution of the LSTM model

loss,accuracy=model.evaluate(X_test,y_test)
print(f"Test Accuracy is: {accuracy*100:.2f}%")

57/57 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.6289 - loss: 0.9569
Test Accuracy is: 62.89%


## Prediction of the unseen data

In [ ]:
# New and unseen raw text data

unseen_tweets=[
    "My new iPhone screen cracked after one drop, absolutely terrible quality!", # Expected: negative
    "Just bought the latest iPad, I am totally in love with the display! @apple", # Expected: positive
    "Google is hosting an event in Texas next month at 5pm.", # Expected: neutral
    "I don't really know how I feel about the new Android update to be honest." # Expected: no_idea / neutral
]

In [ ]:
new_df=pd.DataFrame({'raw_text':unseen_tweets})

In [ ]:
# Cleaning the tweets

import re
def clean_tweet(text):
    text=str(text).lower()
    text=re.sub(r'http\S+|www\S+|https\S+','',text,flags=re.MULTILINE) # Remove URLs
    text=re.sub(r'\@\w+|\#','',text) # Remove mentions & hashtags
    text=re.sub(r'[^a-zA-Z\s]','',text) # Remove punctuation
    return text

In [ ]:
# Cleaning the new text

new_df['cleaned_text']=new_df['raw_text'].apply(clean_tweet)

In [ ]:
# Tokenizing and padding the new sequences

new_sequences=tokenizer.texts_to_sequences(new_df['cleaned_text'])
new_padded=pad_sequences(new_sequences,maxlen=max_sequence_len,padding='post',truncating='post')

In [ ]:
# Generating predictions

prediction_probs=model.predict(new_padded)

1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 577ms/step


In [ ]:
# Getting the index of the highest probability class

predicted_indices=np.argmax(prediction_probs,axis=1)

In [ ]:
# Decoding predictions back to text labels

predicted_labels=encoder.inverse_transform(predicted_indices)

In [ ]:
# Assigning back to the dataframe

new_df['predicted_sentiment']=predicted_labels

In [ ]:
# Mapping back to the original dataset

reverse_mapping={
    "neutral": "No emotion toward brand or product",
    "positive": "Positive emotion",
    "negative": "Negative emotion",
    "no_idea": "I can't tell"
}

In [ ]:
new_df['original_label_format']=new_df['predicted_sentiment'].map(reverse_mapping)

In [ ]:
# Results

for index,row in new_df.iterrows():
    print(f"Prediction: {row['predicted_sentiment'].upper()} ({row['original_label_format']})")

Prediction: POSITIVE (Positive emotion)
Prediction: POSITIVE (Positive emotion)
Prediction: NEUTRAL (No emotion toward brand or product)
Prediction: POSITIVE (Positive emotion)
